# 01 Prediction

## Imports

In [49]:
import numpy as np
from sklearn.metrics import precision_score, recall_score

## What is Prediction? 

Before we dive into the deeper parts of Perception, it is crucial that we understand the types of tasks we have been working with and are going to continue working with. 

**Prediction** is relatively self-explanatory - given some information about the past, we want to determine what is likely going to happen in the future. 

But, there are different ways to perform predictions, based on the types of tasks we are dealing with. You've already seen 2/3 main kinds that we will be dealing with in robotics, but formalizing your understanding of those two (Regression and Classification) will help give us a solid foundation for the 3rd (Object Detection). 

### Evaluating Prediction

When it comes to judging models we should formalize two different contexts: 
1. **Loss** - guides training of the model
2. **Performance** - evaluates the results of the model 

We saw both in action in our previous notebooks. For training our models with cross-validation with the AP Calculus model, we used loss to judge which model configuration worked best. In our CIFAR-10 example, we used performance to determine the best model. 

In practice, our best bet is to find the model that gives us the lowest loss while also maximizing performance! 

### Regression

#### What is it? 

**Regression** is when we predict numerical values. Think back to our AP Calculus example all the way in the ML Primer - we were interested in understanding the exam scores based on hours studied. Our prediction, or outcome, was the exam score, which is a number. 

What are some applications of regression to robotics? 
- Estimating distance to a target (like a hub)
- Predicting our robot speed based on given inputs

So, an abiliity to predict numbers based on inputs is pretty crucial! 

#### Metrics

The most important metric is what we saw in the AP Calculus example - the RMSE (Root Mean Square Error), which quantifies how far off our numerical predictions are from the true value. 

For a refresher, here is what that metric looks like: 

$RMSE = \sqrt{{\frac{1}{n}\sum^n_{i=1}(\hat{y_i} - y_i)^2}}$

Where $\hat{y}$ represents predictions and $y$ represents the true values. 

Note that in that example, we used RMSE for both loss and performance. This is a common theme in regression contexts, since we are predicting numerical values and RMSE works in any context with numbers.

Let's look at an example of that calculation below: 

In [28]:
true_values = np.array([4.0, 6.0, 5.0, 3.0])
predictions = np.array([3.5, 6.3, 5.1, 2.7])

rmse = np.sqrt(np.mean((true_values - predictions) ** 2))
print("RMSE:", round(rmse, 3))

RMSE: 0.332


### Classification

#### What is it? 

**Classification** is when we predict labels/categories, not numerical values. We saw this in our computer vision primer with CIFAR-10, where we were predicting the type of image we were looking at (lizard, car, etc.). 

Classification has some natural robotics applications: 
- Is an object a play piece or another robot?
- Does the object belong to our alliance or another alliance?

Pretty easy to see how classification matters to us! 

#### Metrics

##### Binary Prediction

With classification, it is very common to use **binary predictions**, where our prediction is either right or wrong. How do we quantify this?

Let's go to an example we are familiar with: classifying if the image is an image of a yellow ball. For any image we use, the image is either of a yellow ball, or it isn't. So, we assign any image with a yellow ball a value of 1, and any other image without a yellow ball a value of 0. The 1/0 are also known as Positive/Negative. 

Our predictions therefore also have binary values: 1 if the prediction is yellow ball, 0 if it is not. 

So, we have two possible true values, and two possible predicted values - lets plot this out to see what it could look like: 

![](./ref_imgs/confusion_matrix.png)

We have 4 possible outcomes in the grid above, which is also known as a **confusion matrix**: 
- TN (True Negative): The actual value and predicted value are both 0. That is, we correctly predicted a 0/Negative label.
- FP (False Positive): The actual value is 0, but we predicted 1. So, we incorrectly predicted a 1/Positive label.
- FN (False Negative): The actual value is 1, but we predicted 0. So, we incorreclty labeled a 0/Negative label.
- TP (True Positive): The actual value and predicted value are both 1. That is, we correctly predicted a 1/Positive label.

But how can we turn this into metrics? 

##### Binary Prediction Metrics

We can't really use RMSE here, because our predictions are either right or wrong. 

Note that for loss calculations, a metric called **Binary Cross Entropy Loss** (BCE Loss) is used to quantify how right/wrong our predictions are using probabilities. That formula is given below: 

$$L(y, \hat{y}) = - (y \log(\hat{y}) + (1 - y) \log(1 - \hat{y}))$$

In certain classification tasks, our models output the probability of something belonging to a given label, and we assign the label based on a threshold (e.g., if the probability is above 50%, assign the label). So, this handles loss, but what about performance?

Something like RMSE or BCE Loss doesn't work here, since we are dealing with categories. 

Well, let's use our 4 possible outcomes to derive some metrics below:
- Accuracy: $\frac{TP + TN}{FN + FP + TP + TN}$ - Out of all of our predictions, what proportion did we get correct?
- Precision/True Positive Rate (TPR): $\frac{TP}{TP + FP}$ - Out of our Positive (1) predictions, what proportion did we get correct?
- Recall: $\frac{TP}{TP + FN}$ - Out of our Positive (1) true labels, what proportion did we get correct?
- Specificity/True Negative Rate (TNR): $\frac{TN}{TN + FP}$ - Out of our Negative (0) predictions, what proportion did we get correct?
- F1 Score: $2*\frac{Precision * Recall}{Precision + Recall}$ - Harmonic Mean of Precision and Recall (tries to find best "balance")

Fortunately, calculating these can be used in existing Python packages, like we do below: 

In [94]:
y_true = [1,1,1,0,0,1,0]
y_pred = [1,1,0,0,0,1,1]

print("Precision:", precision_score(y_true, y_pred))
print("Recall:", recall_score(y_true, y_pred))

Precision: 0.75
Recall: 0.75


##### Metric Selection 

So, which one of these do we pick to evaluate performance? Well, why do we even have to pick? Can't we use all of them? 

Without getting too much into the weeds of decision boundaries, take it as true that we cannot increase our Precision while also increasing our Recall. Simply put, if we get more False Positives, by definition, we will have less False Negatives.  

How do we pick between the two? This depends on our use case. If we cannot afford to miss a posiitve label, like in medical contexts, we usually prefer maximizing Recall because this increases the number of TPs we end up identifying correctly. If we are more concerned with getting our positive predictions correct, like with spam emails where we don't want to mark an important email as spam, Precision is the way to go. 

Ultimately, for our robotics use cases, we would probably lean with more balanced metrics like an F1-score because we don't have a convincing argument for picking one over the other. 

Why not accuracy? Accuracy depends on the balance of our dataset! If we have 90% TNs, and 10% TPs, but we predict all of the TPs incorrect, we would still have 90% Accuracy if we get the TNs correct. This is not helpful if we need to be good at detecting positives! 

### Object Detection

#### What is it? 

**Object Detection** is a essentially a combination of Regression and Classification, in that it takes into account two different questions: 
1. What kind of object is it? (Classification)
2. Where in the frame is the object? (Localization, an application of Regression techniques)

The output of object detection in our use case is what we saw during our prep of the 2026 FRC REBUILT Limelight and ball detection data in Roboflow: bounding boxes around the objects! 

For instance, and output for an object detection could look like this: [Object, (x, y)], where Object is the kind of object and (x,y) are the coordinates in a 2D plane. 

We will dive further into object detection and its relevant metrics in later notebooks! 